<a href="https://colab.research.google.com/github/IslamJenishbekov/flats_kg/blob/zhoomart/Flats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib as plt

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("I77/house_kg_flats")

In [ ]:
print(ds)  # Первые 5 записей


DatasetDict({
    train: Dataset({
        features: ['images', 'price', 'url'],
        num_rows: 3071
    })
})


In [ ]:
import os
import json
from datasets import load_dataset

# Загружаем датасет
ds = load_dataset("I77/house_kg_flats")["train"]

# Создаём папку для всех квартир
os.makedirs("flats", exist_ok=True)

# Словарь для хранения данных
grouped_data = {}

flat_counter = 1  # Счётчик для папок

for item in ds:
    url = item["url"]
    images = item["images"]  # Это список объектов PIL.Image

    # Создаём папку для новой квартиры
    folder_name = f"flat{flat_counter}"
    folder_path = os.path.join("flats", folder_name)
    os.makedirs(folder_path, exist_ok=True)

    # Сохраняем изображения
    img_list = []
    for idx, img in enumerate(images):
        if hasattr(img, "save"):  # Проверяем, что это объект PIL.Image
            img_name = f"img{idx + 1}.jpg"
            img_path = os.path.join(folder_path, img_name)

            # Конвертируем в RGB, если у изображения есть прозрачность (RGBA или P)
            if img.mode in ("RGBA", "P"):
                img = img.convert("RGB")

            img.save(img_path, "JPEG")  # Сохраняем изображение
            img_list.append(img_name)

    # Добавляем данные в JSON
    if url not in grouped_data:
        grouped_data[url] = {}

    grouped_data[url][folder_name] = img_list

    flat_counter += 1  # Увеличиваем номер квартиры

# Сохраняем JSON
with open("grouped_flats.json", "w", encoding="utf-8") as f:
    json.dump(grouped_data, f, indent=4, ensure_ascii=False)

print("Готово! JSON создан: grouped_flats.json")


Готово! JSON создан: grouped_flats.json


In [ ]:
print(ds[0])


{'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1159x733 at 0x7F498BFCE310>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=900x900 at 0x7F495CBC9F10>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=652x638 at 0x7F495CBC9110>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1157x732 at 0x7F4957B34990>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1200x703 at 0x7F4957B37F90>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=544x351 at 0x7F4957B348D0>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=637x443 at 0x7F4957B37510>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=625x435 at 0x7F4957B36E10>], 'price': '18 563 600 сом', 'url': 'https://www.house.kg/details/521743066ec024c57fe40-61109241'}


In [ ]:
import json
import base64
from io import BytesIO
from datasets import load_dataset

# Загружаем датасет
ds = load_dataset("I77/house_kg_flats")["train"]

# Словарь для хранения данных
grouped_data = {}

flat_counter = 1  # Счётчик квартир

for item in ds:
    if flat_counter > 200:  # Останавливаем после 200 квартир
        break

    url = item["url"]
    images = item["images"]  # Это список объектов PIL.Image

    img_list = []
    for img in images:
        if hasattr(img, "save"):  # Проверяем, что это объект PIL.Image
            # Конвертируем в RGB (если есть прозрачность)
            if img.mode in ("RGBA", "P"):
                img = img.convert("RGB")

            # Кодируем изображение в Base64
            buffered = BytesIO()
            img.save(buffered, format="JPEG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

            img_list.append(img_base64)

    # Добавляем в JSON
    if url not in grouped_data:
        grouped_data[url] = {}

    grouped_data[url][f"flat{flat_counter}"] = img_list
    flat_counter += 1  # Увеличиваем номер квартиры

# Сохраняем JSON
with open("flats_with_images_200.json", "w", encoding="utf-8") as f:
    json.dump(grouped_data, f, indent=4, ensure_ascii=False)

print("Готово! JSON создан: flats_with_images.json (200 квартир)")


Готово! JSON создан: flats_with_images.json (200 квартир)


In [ ]:
import json
import base64

# Загружаем JSON
with open("flats_with_images_200.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Создаём папку для изображений
import os
os.makedirs("decoded_images", exist_ok=True)

# Декодируем и сохраняем
for url, flats in data.items():
    for flat_name, images in flats.items():
        flat_folder = os.path.join("decoded_images", flat_name)
        os.makedirs(flat_folder, exist_ok=True)  # Создаём папку для квартиры

        for i, img_base64 in enumerate(images):
            img_data = base64.b64decode(img_base64)
            img_path = os.path.join(flat_folder, f"image_{i+1}.jpg")

            with open(img_path, "wb") as img_file:
                img_file.write(img_data)

print("Готово! Изображения сохранены в папке 'decoded_images'")


Готово! Изображения сохранены в папке 'decoded_images'


In [ ]:
import json
import base64
import os

# Загружаем JSON
with open("flats_with_images_200.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Создаём папку для изображений
os.makedirs("decoded_images", exist_ok=True)

# Декодируем и сохраняем, заменяя base64 на пути
for url, flats in data.items():
    for flat_name, images in flats.items():
        flat_folder = os.path.join("decoded_images", flat_name)
        os.makedirs(flat_folder, exist_ok=True)

        new_image_paths = []  # Новый список с путями вместо base64

        for i, img_base64 in enumerate(images):
            img_path = os.path.join(flat_folder, f"image_{i+1}.jpg")
            img_data = base64.b64decode(img_base64)

            with open(img_path, "wb") as img_file:
                img_file.write(img_data)

            new_image_paths.append(img_path)  # Сохраняем путь вместо base64

        data[url][flat_name] = new_image_paths  # Заменяем в JSON

# Сохраняем обновлённый JSON
with open("flats_with_images_200.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Готово! Изображения сохранены, JSON обновлён с путями к файлам.")


Готово! Изображения сохранены, JSON обновлён с путями к файлам.


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/flats_data/train_.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3367 entries, 0 to 3366
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   header_details                  3364 non-null   object 
 1   address                         3367 non-null   object 
 2   latitude                        3364 non-null   float64
 3   longitude                       3364 non-null   float64
 4   user_name                       3367 non-null   object 
 5   user_url                        3367 non-null   object 
 6   tel_number                      3364 non-null   object 
 7   price_dollars                   3367 non-null   float64
 8   Тип предложения                 3364 non-null   object 
 9   Серия                           3364 non-null   object 
 10  Дом                             3364 non-null   object 
 11  Этаж                            3361 non-null   object 
 12  Площадь                         33

In [ ]:
renamed_columns = {
    'header_details': 'header_details',
    'address': 'address',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'user_name': 'user_name',
    'user_url': 'user_url',
    'tel_number': 'telephone',
    'price_dollars': 'price_usd',
    'Тип предложения': 'offer_type',
    'Серия': 'series',
    'Дом': 'house_number',
    'Этаж': 'floor',
    'Площадь': 'area_sq_meters',
    'Отопление': 'heating',
    'Состояние': 'condition',
    'Санузел': 'bathroom_type',
    'Газ': 'gas',
    'Входная дверь': 'front_door',
    'Парковка': 'parking',
    'Высота потолков': 'ceiling_height_meters',
    'Разное': 'miscellaneous',
    'Правоустанавливающие документы': 'legal_documents',
    'views': 'views',
    'hearts': 'likes',
    'publicated': 'published_date',
    'upped': 'last_uplifted',
    'pictures': 'pictures_count',
    'Телефон': 'landline',
    'Интернет': 'internet',
    'Балкон': 'balcony',
    'Мебель': 'furniture',
    'Пол': 'floor_material',
    'Безопасность': 'security_features',
    'Возможность обмена': 'exchange_possible',
    'Возможность рассрочки': 'installment_possible',
    'Возможность ипотеки': 'mortgage_possible',
    'num_of_comments': 'comments_count',
    'Площадь участка': 'land_area_sq_meters',
    'Канализация': 'sewage',
    'Питьевая вода': 'drinking_water',
    'Электричество': 'electricity'
}
df.rename(columns = renamed_columns, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3367 entries, 0 to 3366
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   header_details         3364 non-null   object 
 1   address                3367 non-null   object 
 2   latitude               3364 non-null   float64
 3   longitude              3364 non-null   float64
 4   user_name              3367 non-null   object 
 5   user_url               3367 non-null   object 
 6   telephone              3364 non-null   object 
 7   price_usd              3367 non-null   float64
 8   offer_type             3364 non-null   object 
 9   series                 3364 non-null   object 
 10  house_number           3364 non-null   object 
 11  floor                  3361 non-null   object 
 12  area_sq_meters         3364 non-null   object 
 13  heating                2807 non-null   object 
 14  condition              3081 non-null   object 
 15  bath

In [ ]:
df["header_details"].head(50)

,header_details
0,"2-комн. кв., 47 м2"
1,"2-комн. кв., 42 м2"
2,"3-комн. кв., 110 м2"
3,"2-комн. кв., 68 м2"
4,"1-комн. кв., 46 м2"
5,"3-комн. кв., 101 м2"
6,"3-комн. кв., 84 м2"
7,"4-комн. кв., 83 м2"
8,"1-комн. кв., 18 м2"
9,"3-комн. кв., 83 м2"


In [ ]:
df[['number_of_rooms', 'square']] = df['header_details'].str.split(',', expand=True)
df['number_of_rooms'] = df['number_of_rooms'].str[0]
print(df['number_of_rooms'])

TypeError: cannot convert the series to <class 'int'>

In [4]:
!pip install jupytext
!jupytext --to py /content/drive/MyDrive/flats_data/Flats.ipynb


ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_